### Funciones de Orden Superior
> 1. Las funciones de orden superior son funciones que operan sobre tipos de datos complejos, como: JSON, Arrays y Maps.
> 2. Permiten pasar funciones como argumentos (como expresiones lambda), aplicar transformaciones y devolver Arrays o Maps.
> 3. Son extremadamente útiles para manipular Arrays sin hacerlas explotar.

#### Uso común de funciones de Arrays de orden superior
- TRANSFORM
- FILTER
- EXISTS
- AGGREGATE

#### Sintaxis:
<hr>

`<function_name> (array_column, lambda expression)` <br>
lambda_expression: `element -> expression`

In [0]:
from pyspark.sql import Row

data = [
    (1, ["Keyboard", "monitor", "mouse", "smartphone"]),
    (2, ["tablet", "ram memory", "smartwatch"]),
    (3, ["printer", "laptop"])
]

df_product_items = spark.createDataFrame(
    data,
    ["product_id", "items"]
)
display(df_product_items)

##### 1. Convierte todos los nombres de los elementos a MAYÚSCULAS (Función TRANSFORM)

In [0]:
from pyspark.sql.functions import transform, upper

df_result = (
    df_product_items
    .select(
        "product_id",
        transform("items", lambda x: upper(x)).alias("upper_items")
    )
)
display(df_result)

##### 2. Filtrar solo los elementos que contengan la cadena 'smart' (Función FILTER)

In [0]:
from pyspark.sql.functions import filter

df_result = (
    df_product_items
    .select(
        "product_id",
        filter("items", lambda x: x.like("%smart%")).alias("smart_items")
    )
)
display(df_result)

##### 3. Comprobar si existe un producto 'tablet' (función EXISTS).

In [0]:
from pyspark.sql.functions import col, exists

df_result = (
    df_product_items
    .select(
        "product_id",
        exists("items", lambda x: x == "tablet").alias("has_tablet")
    )
)
display(df_result)

#### Array con más de un objeto

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType

# Definir el esquema del struct dentro del array
item_schema = StructType([
    StructField("name", StringType(), True),
    StructField("price", IntegerType(), True)
])

# Esquema completo
schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("items", ArrayType(item_schema), True)
])

# Datos
data = [
    (
        1,
        [
            {"name": "Keyboard", "price": 99},
            {"name": "monitor", "price": 899},
            {"name": "mouse", "price": 169},
            {"name": "smartphone", "price": 799},
        ]
    ),
    (
        2,
        [
            {"name": "tablet", "price": 359},
            {"name": "ram memory", "price": 89},
            {"name": "smartwatch", "price": 329},
        ]
    ),
    (
        3,
        [
            {"name": "printer", "price": 299},
            {"name": "laptop", "price": 1299},
        ]
    )
]

# Crear DataFrame
df_product_items = spark.createDataFrame(data, schema)
display(df_product_items)

##### 1. Convertir todos los nombres de los artículos a MAYÚSCULAS y agregue un 18% de IMPUESTO a cada artículo (Función TRANSFORM).

In [0]:
from pyspark.sql.functions import transform, upper, round

df_result = (
    df_product_items
    .select(
        "product_id",
        transform(
            "items",
            lambda x: (
                x
                .withField("name", upper(x["name"]))
                .withField("price", round(x["price"] * 1.18, 2))
            )
        ).alias("items_with_tax")
    )
)
display(df_result)

##### 2. Calcular el "importe total" del producto para cada "product_id" (Función AGGREGATE)

In [0]:
from pyspark.sql.functions import aggregate, lit

df_result = (
    df_product_items
    .select(
        "product_id",
        aggregate(
            "items",
            lit(0),
            lambda acc, x: acc + x["price"]
        ).alias("total_product_price")
    )
)
display(df_result)

#### Función Maps
<hr>

> Un mapa es una colección de pares clave-valor, como un diccionario. <br>`{'smartphone': 1100, 'tablet': 600}`

#### Uso común de funciones "maps" de orden superior
- TRANSFORM_VALUES
- TRANSFORM_KEYS
- MAP_FILTER

#### Sintaxis:
<hr>

`<function_name> (map_column, lambda_expression)` <br>
_lambda expression:_ `(key_value) -> expression`

In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F

data = [
    (1, {
        "Keyboard": 99,
        "monitor": 899,
        "mouse": 169,
        "smartphone": 799
    }),
    (2, {
        "tablet": 359,
        "ram memory": 89,
        "smartwatch": 329
    }),
    (3, {
        "printer": 299,
        "laptop": 1299
    })
]

product_item_prices = spark.createDataFrame(
    data,
    ["product_id", "item_prices"]
)

display(product_item_prices)

##### 1. Convierte todos los nombres de los elementos a MAYÚSCULAS (Función TRANSFORM_KEYS)

In [0]:
from pyspark.sql.functions import transform_keys, upper

result_df = (
    product_item_prices
        .select(
            "product_id",
            transform_keys(
                "item_prices",
                lambda item, price: upper(item)
            ).alias("items_upper_case")
        )
)
display(result_df)

##### 2. Aplicar un 18% de IMPUESTO a los precios de los artículos (Función TRANSFORM_VALUES)

In [0]:
from pyspark.sql.functions import transform_values, round

result_df = (
    product_item_prices
        .select(
            "product_id",
            transform_values(
                "item_prices",
                lambda item, price: round(price * 1.18, 2)
            ).alias("items_with_tax")
        )
)
display(result_df)

##### 3. Filtrar solo los artículos con un precio superior a 700 dólares (Función MAP_FILTER)

In [0]:
from pyspark.sql.functions import map_filter

result_df = (
    product_item_prices
        .select(
            "product_id",
            map_filter(
                "item_prices",
                lambda item, price: price > 700
            ).alias("primiun_items")
        )
)
display(result_df)